# 09 — Evaluation: Kupiec POF Test and VaR Backtesting

**What:** Formally evaluate the conformal prediction intervals and the
GARCH-based Value-at-Risk using two industry-standard backtesting methods:

1. **Kupiec Proportion of Failures (POF) test** — tests whether the
   empirical violation rate of the conformal intervals is statistically
   consistent with the nominal rate.
2. **VaR backtesting with the Basel traffic light test** — converts GARCH
   conditional volatility into a 99% Value-at-Risk estimate and counts
   exceptions against regulatory thresholds.

**Why:** The coverage check in notebook 08 showed empirical coverage at
all three levels — a necessary condition but not sufficient for operational
use. Kupiec's POF test asks whether the violation rate is statistically
consistent with the nominal rate, distinguishing genuine coverage from
lucky sampling. The VaR test connects the volatility forecast to
regulatory capital requirements — the ultimate use case for this pipeline.

**How:** All evaluation logic is implemented inline. The clean version
will consolidate these functions into `src/models/evaluation.py`.

**Outline:**

0. Setup
1. Kupiec POF Test — Theory
2. Kupiec POF Test — Results
3. VaR Backtesting — Theory
4. VaR Backtesting — Results
5. Regime-Conditioned Coverage
6. Summary
7. Conclusion


---
---
## 0. Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import t as t_dist

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.data.describe import compute_returns
from src.models.garch import VolatilityModel

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12, 'axes.labelsize': 11})

# conformal intervals from notebook 08
intervals = pd.read_csv(
    ROOT / 'data/processed/conformal_intervals.csv',
    index_col='Date', parse_dates=True
)

# raw returns for VaR backtesting (signed, not absolute)
mxn     = pd.read_csv(ROOT / 'data/raw/mxn_usd.csv', index_col='Date', parse_dates=True)
returns = compute_returns(mxn['MXN_USD'])

# HMM regimes for regime-conditioned coverage
regimes = pd.read_csv(
    ROOT / 'data/processed/hmm_regimes.csv',
    index_col='Date', parse_dates=True
)

# align to test set index
test_returns = returns.reindex(intervals.index)
test_regimes = regimes.reindex(intervals.index)

print(f'Test set  : {intervals.shape[0]:,} days | '
      f'{intervals.index[0].date()} → {intervals.index[-1].date()}')
print(f'NaN check : returns={test_returns.isna().sum()}, '
      f'regimes={test_regimes.isna().sum().values[0]}')


---
---
## 1. Kupiec POF Test — Theory

The **Proportion of Failures (POF) test** (Kupiec, 1995) tests whether
the observed violation rate $\hat{p}$ is statistically consistent with
the nominal violation rate $p$.

A violation at time $t$ occurs when the realised value exceeds the
upper bound of the prediction interval:
$$I_t = \mathbf{1}[y_t > U_t]$$

Under H0, violations are i.i.d. Bernoulli($p$), so the number of
violations $V = \sum_t I_t$ follows Binomial($n$, $p$). The
likelihood ratio test statistic is:
$$LR_{\text{POF}} = -2\log\!\left[\frac{(1-p)^{n-V} p^V}
{(1-\hat{p})^{n-V}\hat{p}^V}\right] \sim \chi^2(1)$$

where $\hat{p} = V/n$. A p-value below 0.05 means the violation rate
is statistically inconsistent with the nominal rate.

**Asymmetric conformal correction.** The production model
(`AsymmetricConformalPredictor`, $\phi = 0.7$) splits the total error
budget $\alpha$ asymmetrically: the upper tail receives fraction
$\phi$ of the budget and the lower tail receives $1-\phi$.  Therefore
the correct null hypothesis for the upper-bound Kupiec test is not
$\alpha$ but:
$$p = \alpha_{\text{upper}} = \alpha \times \phi$$

| Interval | $\alpha$ | $\alpha_{\text{upper}} = \alpha \times 0.7$ |
|----------|----------|----------------------------------------------|
| 80%      | 0.20     | 0.14                                         |
| 90%      | 0.10     | 0.07                                         |
| 95%      | 0.05     | 0.035                                        |

**One-sided risk interpretation:** undercoverage ($\hat{p} > \alpha_{\text{upper}}$)
is the dangerous failure — intervals are too narrow and underestimate
tail risk. Overcoverage ($\hat{p} < \alpha_{\text{upper}}$) means conservative
intervals — safe but capital-inefficient.


---
---
## 2. Kupiec POF Test — Results

In [ ]:
def kupiec_pof(y_true, upper, alpha):
    """Kupiec POF test. Violation = y_true > upper bound."""
    n          = len(y_true)
    violations = np.sum(y_true > upper)
    p_hat      = violations / n
    eps        = 1e-10
    p_hat_s    = np.clip(p_hat, eps, 1 - eps)
    alpha_s    = np.clip(alpha, eps, 1 - eps)
    lr_stat    = -2 * (
        (n - violations) * np.log((1 - alpha_s) / (1 - p_hat_s)) +
        violations       * np.log(alpha_s / p_hat_s)
    )
    p_value = 1 - stats.chi2.cdf(lr_stat, df=1)
    return {'violations': int(violations), 'n': n,
            'p_hat': float(p_hat), 'lr_stat': float(lr_stat),
            'p_value': float(p_value)}

# Asymmetry parameter (phi) — must match params.yaml conformal.asymmetry
PHI = 0.7

y_true         = intervals['y_true'].values
kupiec_results = {}

for alpha, col in [(0.20, 'upper_80'), (0.10, 'upper_90'), (0.05, 'upper_95')]:
    # Correct null hypothesis for asymmetric conformal: upper tail gets phi * alpha
    alpha_upper = alpha * PHI
    upper       = intervals[col].values
    result      = kupiec_pof(y_true, upper, alpha_upper)
    result['alpha_upper'] = alpha_upper          # store for display
    kupiec_results[f'{int((1-alpha)*100)}%'] = result

kupiec_df = pd.DataFrame({
    'Nominal Level'         : list(kupiec_results.keys()),
    'α_upper (null)'        : [f'{v["alpha_upper"]:.1%}'   for v in kupiec_results.values()],
    'Violations'            : [v['violations']              for v in kupiec_results.values()],
    'n'                     : [v['n']                       for v in kupiec_results.values()],
    'Emp. Viol. Rate'       : [f'{v["p_hat"]:.2%}'          for v in kupiec_results.values()],
    'LR Statistic'          : [f'{v["lr_stat"]:.4f}'        for v in kupiec_results.values()],
    'p-value'               : [f'{v["p_value"]:.4f}'        for v in kupiec_results.values()],
    'Reject H0 (5%)'        : ['Yes' if v['p_value'] < 0.05 else 'No'
                                for v in kupiec_results.values()],
}).set_index('Nominal Level')

display(kupiec_df)


---
---
## 3. VaR Backtesting — Theory

**Value-at-Risk (VaR)** at confidence level $1-\alpha$ is the loss
not exceeded with probability $1-\alpha$:
$$P(r_t < -\text{VaR}_t) = \alpha$$

For a GARCH-$t$ model, the 99% one-day VaR is:
$$\text{VaR}_t^{99\%} = -(\hat{\mu} + \hat{\sigma}_t \cdot t_{\nu}^{-1}(0.01))$$

where $\hat{\sigma}_t$ is the daily GARCH conditional volatility,
$t_{\nu}^{-1}(0.01)$ is the 1st percentile of the Student-$t$
distribution with $\hat{\nu}$ degrees of freedom, and $\hat{\mu}$
is the fitted mean return.

**Basel traffic light test** — over 250 trading days:

| Exceptions | Zone   | Consequence |
|-----------|--------|-------------|
| 0–4       | Green  | No action |
| 5–9       | Yellow | Supervisory review |
| 10+       | Red    | Capital add-on required |

Thresholds are scaled proportionally to the test set length.


---
---
## 4. VaR Backtesting — Results

In [ ]:
garch = VolatilityModel.load(ROOT / 'data/processed/garch.pkl')

garch_vol_daily = garch.conditional_volatility() / np.sqrt(252)
nu  = garch.summary()['nu']
mu  = garch.summary()['mu']

garch_daily_test = garch_vol_daily.reindex(intervals.index)
t_quantile       = t_dist.ppf(0.01, df=nu)
var_99           = -(mu + garch_daily_test.values * t_quantile)

actual_returns = test_returns.values
exceptions     = actual_returns < -var_99
n_exceptions   = int(np.sum(exceptions))
exception_rate = n_exceptions / len(actual_returns)

scale  = len(actual_returns) / 250
green  = int(4 * scale)
yellow = int(9 * scale)
zone   = 'Green' if n_exceptions <= green else ('Yellow' if n_exceptions <= yellow else 'Red')

print(f'Test period       : {len(actual_returns):,} days')
print(f'Exceptions        : {n_exceptions}')
print(f'Exception rate    : {exception_rate:.2%}  (expected 1.00%)')
print(f'Basel zone        : {zone}')
print(f'Scaled thresholds : Green ≤ {green}, Yellow ≤ {yellow}, Red > {yellow}')


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(intervals.index, actual_returns,
        color='steelblue', linewidth=0.6, label='Actual return')
ax.plot(intervals.index, -var_99,
        color='crimson', linewidth=1.0, linestyle='--', label='−VaR 99%')
ax.scatter(intervals.index[exceptions], actual_returns[exceptions],
           color='crimson', s=20, zorder=5, label=f'Exceptions ({n_exceptions})')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('GARCH-t 99% VaR Backtesting — Test Set (2022–2026)')
ax.set_ylabel('Daily Log-return')
ax.legend()
plt.tight_layout()
plt.show()


---
---
## 5. Regime-Conditioned Coverage

The conformal intervals have constant width — they do not adapt to the
current volatility regime. This section asks whether coverage holds
equally across all three HMM regimes, or whether it fails in specific
market states.

**Target for the upper-bound violation rate.** Because the production
model is asymmetric ($\phi = 0.7$), the expected upper violation rate
for a 90% interval is not 10% but:
$$\alpha_{\text{upper}} = 0.10 \times 0.70 = 7\%$$

We therefore compare per-regime empirical upper violation rates against
this 7% target. Systematic undercoverage in the high-volatility regime
($\hat{p} \gg 7\%$) would mean the fixed-width interval is insufficient
precisely when tail risk is largest.


In [ ]:
regime_labels = {0: 'Low', 1: 'Medium', 2: 'High'}

# Upper violation target for asymmetric conformal at 90% level
ALPHA_UPPER_90 = 0.10 * PHI   # = 0.07

rows = []
for k in range(3):
    mask = test_regimes['regime'].values == k
    if mask.sum() == 0:
        continue
    y_t      = y_true[mask]
    u_90     = intervals['upper_90'].values[mask]
    viol_rate = np.mean(y_t > u_90)
    viol      = int(np.sum(y_t > u_90))
    rows.append({
        'Regime'                 : regime_labels[k],
        'n'                      : int(mask.sum()),
        'Upper Violations'       : viol,
        'Emp. Viol. Rate'        : f'{viol_rate:.2%}',
        'Target (α_upper = 7%)' : '7.00%',
        'Gap vs target'          : f'{viol_rate - ALPHA_UPPER_90:+.2%}',
    })

regime_df = pd.DataFrame(rows).set_index('Regime')
display(regime_df)


---
---
## 6. Summary

In [ ]:
print('=' * 65)
print('EVALUATION SUMMARY')
print('=' * 65)
print()
print('Kupiec POF Test (conformal intervals):')
display(kupiec_df)
print()
print('VaR Backtesting (GARCH-t 99%):')
print(f'  Exceptions    : {n_exceptions} / {len(actual_returns):,} days')
print(f'  Exception rate: {exception_rate:.2%}  (expected 1.00%)')
print(f'  Basel zone    : {zone}')
print()
print('Regime-conditioned coverage (90% interval):')
display(regime_df)


---
---
## 7. Conclusion

**Kupiec POF test — asymmetric null hypothesis.** The production model
is `AsymmetricConformalPredictor` with $\phi = 0.7$, so the correct
null for the upper-bound Kupiec test is $\alpha_{\text{upper}} = \alpha
\times 0.7$, not $\alpha$ itself. The tests evaluate whether the
empirical upper violation rate is consistent with 14% (80% interval),
7% (90%), and 3.5% (95%) respectively — not with 20%, 10%, and 5%.
Applying the correct null, H0 is not rejected at any level, confirming
that the asymmetric upper bounds are well-calibrated: the model
deliberately places more of the error budget on the upside tail where
volatility shocks are most consequential.

**VaR backtesting.** The GARCH-$t$ 99% VaR produces zero exceptions
over the test period — an exception rate of 0.00% against an expected
1.00%. The model lands firmly in the Basel green zone. Zero exceptions
over nearly four years of out-of-sample data means the GARCH-$t$ VaR
is extremely conservative — it overestimates tail risk, producing VaR
estimates that are never breached. This is consistent with the
aggregate picture: the pipeline systematically overestimates volatility
on this test period, producing intervals and VaR estimates that are
safe but capital-inefficient.

**Regime-conditioned coverage.** The 90% conformal interval is
evaluated against the asymmetric upper-violation target of 7%
($= 10\% \times 0.7$). In the low-volatility regime the interval is
massively conservative — far fewer than 7% of days breach the upper
bound. In the medium regime the empirical rate is close to the 7%
target, showing good calibration during normal conditions. In the
high-volatility (crisis) regime the upper violation rate exceeds the
7% target by a meaningful margin, revealing that the fixed-width
interval is still insufficiently wide during crises despite the
asymmetric design pushing more budget upward.

**The key finding.** The asymmetric conformal design ($\phi = 0.7$)
correctly re-allocates error budget toward the upper tail, and passes
the aggregate Kupiec test at all three coverage levels. The remaining
weakness is regime-dependent: during the small fraction of days
classified as high-volatility, even the widened upper bound is
insufficient. This is the strongest empirical argument for locally
adaptive conformal prediction as a future extension — an interval that
additionally scales with the current HMM regime would improve both
capital efficiency in calm periods and crisis coverage simultaneously.
The regime-conditioned violation table provides the exact quantitative
motivation for that extension.

**Pipeline assessment.** The full pipeline — GARCH/EGARCH for
continuous volatility, HMM for regime detection, XGBoost hybrid for
point forecasting, and asymmetric split conformal ($\phi = 0.7$) for
intervals — is well-calibrated in aggregate and passes formal Kupiec
tests when the correct (asymmetric) null hypothesis is applied. The
actionable conclusion for a risk manager: the asymmetric 90% upper
bound is reliable on most trading days, but should be treated with
caution during the minority of days classified as the high-volatility
regime, where the empirical breach rate exceeds the 7% design target.
